In [1]:
import os
import sys
import platform
import torch

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("CPU cores:", os.cpu_count())
print("CUDA available:", torch.cuda.is_available())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

Python: 3.13.16
Platform: Linux-6.18.55+-x86_64-with-glibc2.39
PyTorch: 2.11.0+cu130
CUDA runtime: 13.0
CPU cores: 4
CUDA available: True
GPU 0: Tesla T4
GPU 1: Tesla T4


In [2]:
from pathlib import Path
from kaggle_secrets import UserSecretsClient
import subprocess
import tempfile
import os

ROOT = Path("/kaggle/working/GATP-MAIN")
REPO = "https://github.com/JLiang627/GATP-MAIN.git"

token = UserSecretsClient().get_secret("GITHUB_TOKEN")

with tempfile.TemporaryDirectory() as tmp:
    askpass = Path(tmp) / "askpass.sh"

    askpass.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        '  *Username*) printf "%s\\n" "x-access-token" ;;\n'
        '  *Password*) printf "%s\\n" "$GITHUB_TOKEN" ;;\n'
        'esac\n'
    )
    askpass.chmod(0o700)

    env = os.environ.copy()
    env["GITHUB_TOKEN"] = token
    env["GIT_ASKPASS"] = str(askpass)
    env["GIT_TERMINAL_PROMPT"] = "0"
    env["GIT_CONFIG_GLOBAL"] = os.devnull
    env["GIT_CONFIG_NOSYSTEM"] = "1"

    if ROOT.exists():
        if (ROOT / ".git").exists():
            print("Repository already exists.")
        elif not any(ROOT.iterdir()):
            ROOT.rmdir()
        else:
            raise RuntimeError(
                "Incomplete clone found. Inspect the directory before retrying."
            )

    if not ROOT.exists():
        subprocess.run(
            [
                "git",
                "-c", "credential.helper=",
                "clone",
                REPO,
                str(ROOT)
            ],
            env=env,
            check=True
        )

print("Clone completed:", ROOT)

Cloning into '/kaggle/working/GATP-MAIN'...


Clone completed: /kaggle/working/GATP-MAIN


In [3]:
from pathlib import Path
import subprocess

ROOT = Path("/kaggle/working/GATP-MAIN")

def git(*args):
    return subprocess.check_output(
        ["git", *args],
        cwd=ROOT,
        text=True
    ).strip()

assert (ROOT / ".git").exists(), "Repository not cloned"

print("Repository:", ROOT)
print("Branch:", git("branch", "--show-current") or "DETACHED")
print("Commit:", git("rev-parse", "HEAD"))

print("\nAvailable branches:")
print(git("branch", "-a"))

print("\nProject directories:")
for name in ["src", "experiments", "thesis", "third_party"]:
    print(f"{name}: {(ROOT / name).exists()}")

Repository: /kaggle/working/GATP-MAIN
Branch: main
Commit: dee1cdcd8b71338613d1a22552f37e4f9e2ef11b

Available branches:
* main
  remotes/origin/HEAD -> origin/main
  remotes/origin/agent/dev
  remotes/origin/analysis/intent-v1-architecture
  remotes/origin/audit/agent-c-abc-frozen-impl
  remotes/origin/audit/agent-c-model-native-independent
  remotes/origin/audit/intent-subgoal-v1
  remotes/origin/audit/intent-subgoal-v1-round2
  remotes/origin/audit/intent-v1-diag-oracle-bf3
  remotes/origin/audit/intent-v1-eval-reliability
  remotes/origin/audit/intent-v1-learning-pipeline
  remotes/origin/audit/intent-v1-overnight-gate
  remotes/origin/audit/intent-v1-round3
  remotes/origin/audit/neural-only-v2
  remotes/origin/audit/original-gatp-minimal-changes
  remotes/origin/claude/friendly-khorana-afebb4
  remotes/origin/diag/intent-v1-goal-intent
  remotes/origin/diag/neural-only-v2
  remotes/origin/docs/g27-scenario-visualization
  remotes/origin/docs/model-native-research-index
  remotes/

In [4]:
import importlib
import importlib.util

packages = [
    "torch",
    "numpy",
    "scipy",
    "pandas",
    "matplotlib",
    "networkx",
    "cvxpy",
    "osqp",
    "torch_geometric",
    "pytest",
]

for pkg in packages:
    try:
        spec = importlib.util.find_spec(pkg)
        if spec is None:
            print(f"MISSING: {pkg}")
            continue

        mod = importlib.import_module(pkg)
        version = getattr(mod, "__version__", "unknown")
        print(f"OK: {pkg} ({version})")

    except Exception as e:
        print(f"ERROR: {pkg}: {type(e).__name__}: {e}")

OK: torch (2.11.0+cu130)
OK: numpy (2.1.3)
OK: scipy (1.16.3)
OK: pandas (2.3.3)
OK: matplotlib (3.10.0)
OK: networkx (3.7)
OK: cvxpy (1.6.7)
OK: osqp (1.1.3)
MISSING: torch_geometric
OK: pytest (8.4.2)


In [5]:
import torch

assert torch.cuda.is_available(), "CUDA unavailable"

for i in range(torch.cuda.device_count()):
    device = torch.device(f"cuda:{i}")

    x = torch.randn(256, 256, device=device)
    y = torch.matmul(x, x)

    torch.cuda.synchronize(i)

    print(f"GPU {i}: PASS")
    print("Device:", torch.cuda.get_device_name(i))
    print("Result finite:", torch.isfinite(y).all().item())

    del x, y

torch.cuda.empty_cache()

GPU 0: PASS
Device: Tesla T4
Result finite: True
GPU 1: PASS
Device: Tesla T4
Result finite: True


In [6]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess
import platform
import torch
import json
import sys

WORK = Path("/kaggle/working/gatp_workspace")

for folder in ["outputs", "logs", "checkpoints", "artifacts"]:
    (WORK / folder).mkdir(parents=True, exist_ok=True)

commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=ROOT,
    text=True
).strip()

info = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "repository": "JLiang627/GATP-MAIN",
    "commit": commit,
    "python": sys.version,
    "platform": platform.platform(),
    "pytorch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu_names": [
        torch.cuda.get_device_name(i)
        for i in range(torch.cuda.device_count())
    ],
    "training_started": False,
}

with open(WORK / "environment.json", "w") as f:
    json.dump(info, f, indent=2)

print("Workspace ready:", WORK)
print("Repository commit:", commit)
print("Training started: NO")

Workspace ready: /kaggle/working/gatp_workspace
Repository commit: dee1cdcd8b71338613d1a22552f37e4f9e2ef11b
Training started: NO


In [7]:
from pathlib import Path
import subprocess

ROOT = Path("/kaggle/working/GATP-MAIN")

# 未來要執行哪個實驗，再填入它的完整 Commit SHA。
TARGET_COMMIT = ""

if not TARGET_COMMIT:
    print("No research commit selected.")
    print("Current environment remains unchanged.")
else:
    status = subprocess.check_output(
        ["git", "status", "--porcelain"],
        cwd=ROOT,
        text=True
    )

    if status.strip():
        raise RuntimeError(
            "Working tree has uncommitted changes."
        )

    subprocess.run(
        ["git", "checkout", "--detach", TARGET_COMMIT],
        cwd=ROOT,
        check=True
    )

    actual = subprocess.check_output(
        ["git", "rev-parse", "HEAD"],
        cwd=ROOT,
        text=True
    ).strip()

    if actual != TARGET_COMMIT:
        raise RuntimeError("Commit mismatch")

    print("Selected research commit:", actual)

No research commit selected.
Current environment remains unchanged.
